In [1]:
import sqlite3
from pathlib import Path
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
raw = sqlite3.connect(ROOT / "data/goszakup_raw.db")
clean = sqlite3.connect(ROOT / "data/goszakup_clean.db")

def q(sql, db=clean):
    return pd.read_sql(sql, db)

units = q("""SELECT json_extract(payload,'$.id') AS id,
                    json_extract(payload,'$.contract_registry_id') AS contract_id,
                    json_extract(payload,'$.pln_point_id') AS plan_id,
                    json_extract(payload,'$.lot_id') AS lot_id,
                    json_extract(payload,'$.deleted') AS deleted
             FROM raw WHERE entity='contract_unit'""", raw)
contracts = q("SELECT contract_id, root_id, year, is_latest, deleted FROM contracts")
lines = q("SELECT * FROM contract_lines")
print(len(units), len(contracts), len(lines))

105166 37039 78275


In [2]:
m = units.merge(contracts, on="contract_id", how="left")

has_units = set(units.loc[units.deleted.fillna(0) == 0, "contract_id"])
latest = contracts[contracts.is_latest == 1].set_index("root_id").contract_id
latest_has_units = latest.map(lambda cid: cid in has_units)

def category(r):
    if r.deleted_x == 1:            return "1. тауар жолы өшірілген"
    if pd.isna(r.root_id):          return "2. шарт табылмады"
    if r.deleted_y == 1:            return "3. шарт өшірілген"
    if r.is_latest == 1:            return "0. сақталды"
    if latest_has_units.get(r.root_id, False):
                                    return "4. ескі нұсқа, жаңасында жолдар бар (қалыпты)"
    return                                 "5. ескі нұсқа, жаңасында жол ЖОҚ (жоғалған)"

m["cat"] = m.apply(category, axis=1)
print(m.cat.value_counts().sort_index())

# Соңғы нұсқасында бірде-бір тауар жолы жоқ шарттар
lc = contracts[(contracts.is_latest == 1) & (contracts.deleted.fillna(0) == 0)]
lc = lc.assign(no_lines=~lc.contract_id.isin(has_units))
print("\nСоңғы нұсқасында жолы жоқ шарттар, жылдар бойынша:")
print(lc.groupby("year").no_lines.agg(["sum", "size"]))

cat
0. сақталды                                      78275
4. ескі нұсқа, жаңасында жолдар бар (қалыпты)    26890
5. ескі нұсқа, жаңасында жол ЖОҚ (жоғалған)          1
Name: count, dtype: int64

Соңғы нұсқасында жолы жоқ шарттар, жылдар бойынша:
      sum   size
year            
2024    0  11226
2025    0  10683
2026    2   8917


In [3]:
print(q("SELECT entity, COUNT(*) AS n FROM raw GROUP BY entity ORDER BY n DESC", raw))

missing = lines[lines.plan_id.isna()]
miss_ids = units[units.id.isin(missing.line_id)].plan_id.dropna().astype(int).unique()
print(f"\nТабылмаған тауар жолдары: {len(missing)}, бірегей жоспар ID: {len(miss_ids)}")
print(missing.groupby("year").size())

raw_ids = set(q("SELECT obj_id FROM raw WHERE entity IN ('plan','plan_hist')", raw)
              .obj_id.astype(str))
in_raw = sum(str(i) in raw_ids for i in miss_ids)
print(f"\nОлардың шикі дерекқорда бары: {in_raw} / {len(miss_ids)}")

            entity       n
0              lot  116368
1    contract_unit  105166
2             plan   74100
3     trd_buy_skip   38045
4         contract   37039
5          trd_buy   36488
6              rnu   17291
7          subject    8974
8    plan_hist_404    5543
9        plan_hist    4633
10      rnu_detail    2636
11     subject_404     256
12  rnu_detail_404     113
13            done      27
14             ref      14

Табылмаған тауар жолдары: 6457, бірегей жоспар ID: 6332
year
2024     353
2025     355
2026    5749
dtype: int64

Олардың шикі дерекқорда бары: 0 / 6332


In [4]:
lots = q("SELECT lot_id, region_code, kato_code FROM lots")
pd_ = q("SELECT plan_id, region_code FROM plan_delivery")
n_regions = pd_.dropna().drop_duplicates().groupby("plan_id").size()

nr = lines[lines.region_code.isna()].merge(lots, on="lot_id", how="left", suffixes=("", "_lot"))

def region_reason(r):
    if pd.isna(r.lot_id) or r.lot_id == 0:  lot = "лот жоқ"
    elif pd.isna(r.kato_code):              lot = "лотта КАТО жоқ"
    else:                                   lot = "лот бар, өңірі анықталмады"
    k = n_regions.get(r.plan_id, 0)
    plan = "жоспарда өңір жоқ" if k == 0 else f"жоспарда {k} өңір"
    return f"{lot} | {plan if k < 2 else 'жоспарда бірнеше өңір'}"

nr["reason"] = nr.apply(region_reason, axis=1)
print(f"Өңірі жоқ жолдар: {len(nr)}")
print(nr.reason.value_counts())

Өңірі жоқ жолдар: 24142
reason
лотта КАТО жоқ | жоспарда өңір жоқ    19605
лот жоқ | жоспарда өңір жоқ            4525
лот жоқ | жоспарда бірнеше өңір          12
Name: count, dtype: int64


In [5]:
import json

# Табылмаған жолдардың бастапқы plan_id-імен
miss = (lines[lines.plan_id.isna()][["line_id", "lot_id", "year"]]
        .merge(units[["id", "plan_id"]].rename(columns={"id": "line_id", "plan_id": "orig_id"}),
               on="line_id"))
miss["orig_id"] = pd.to_numeric(miss.orig_id, errors="coerce").astype("Int64")

# A) 404 қайтарғандар
h404 = set(q("SELECT obj_id FROM raw WHERE entity='plan_hist_404'", raw).obj_id.astype(str))
print("A) 404 қайтарған:", miss.orig_id.astype(str).isin(h404).sum(), "/", len(miss))

# B) Жаңа нұсқа root_id арқылы
pi = q("SELECT plan_id, root_id, enstru_code FROM plan_items")
pi["root_id"] = pd.to_numeric(pi.root_id, errors="coerce").astype("Int64")
pi["plan_id"] = pd.to_numeric(pi.plan_id, errors="coerce").astype("Int64")
ok_root = miss.orig_id.isin(pi.root_id)
print("B) root_id арқылы табылады:", ok_root.sum())

# C) Лоттың point_list арқылы
lp = q("SELECT lot_id, point_list FROM lots")
print("\npoint_list мысалдары:", lp.point_list.dropna().head(3).tolist())
lp["pid"] = lp.point_list.astype(str).str.findall(r"\d{5,}")
ex = lp.explode("pid").dropna(subset=["pid"])
ex["pid"] = ex.pid.astype("int64")
ok_lots = set(ex[ex.pid.isin(pi.plan_id.dropna())].lot_id)
ok_lot = miss.lot_id.isin(ok_lots)
print("C) лоттың point_list арқылы табылады:", ok_lot.sum())
print("B немесе C:", (ok_root | ok_lot).sum(), "/", len(miss))

# D) Шикі лотта КАТО немесе мекенжай өрістері бар ма
no_kato = lots[lots.kato_code.isna()].lot_id.head(1).tolist()
p = json.loads(q(f"SELECT payload FROM raw WHERE entity='lot' AND obj_id='{no_kato[0]}'", raw).payload[0])
print("\nD) КАТО-сыз лоттың өрістері:")
print({k: v for k, v in p.items() if any(s in k.lower() for s in ("kato", "place", "deliv", "addr", "point"))})

# E) Тапсырыс берушінің мекенжайы subject-те бар ма
s = json.loads(q("SELECT payload FROM raw WHERE entity='subject' LIMIT 1", raw).payload[0])
print("\nE) subject өрістері:")
print({k: v for k, v in s.items() if any(x in k.lower() for x in ("kato", "addr", "region", "bin"))})

A) 404 қайтарған: 5542 / 6457
B) root_id арқылы табылады: 320

point_list мысалдары: ['[64271375, 64271734, 64271958, 64272016, 64306083, 64314885, 64315186, 64380806, 64388811, 64388821, 64388842, 64388897, 64388909, 64388918]', '[64272394, 64272635, 64272685, 64272861, 64272900, 64272938, 64273006, 64273060, 64273110, 64273176, 64273224, 64273271, 64273331, 64273797, 64274095, 64274143, 64274191, 64274230, 64282409, 64282471, 64282543, 64282651, 64282698, 64282768, 64282815, 64282861, 64282899, 64282939, 64282977, 64283077, 64289794, 64289883, 64289941, 64289984, 64290011, 64290032, 64290059, 64290085, 64290681, 64290752, 64290798, 64291893, 64291958, 64292559, 64292603, 64292668, 64292706, 64292742, 64292797, 64292841]', '[64266911, 64266958, 64266996, 64267027, 64267059, 64308868, 64308991, 64309043, 64309111, 64309152, 64315265, 64315321, 64315363, 64315413, 64379911, 64380348]']
C) лоттың point_list арқылы табылады: 988
B немесе C: 1017 / 6457

D) КАТО-сыз лоттың өрістері:
{'poin

In [6]:
# 1) Лоттың жоспар тармақтарында ЕНС ТРУ мен өлшем бірлігі бірдей ме
pi2 = q("SELECT plan_id, enstru_code, unit_code FROM plan_items")
pi2["plan_id"] = pd.to_numeric(pi2.plan_id, errors="coerce").astype("Int64")
ex["pid"] = ex.pid.astype("Int64")
lot_items = ex.merge(pi2, left_on="pid", right_on="plan_id")
k = lot_items.groupby("lot_id").agg(n_enstru=("enstru_code", "nunique"),
                                    n_unit=("unit_code", "nunique"))
need = k[k.index.isin(miss.lot_id)]
safe = need[(need.n_enstru == 1) & (need.n_unit == 1)]
print(f"1) Керекті лоттар: {len(need)}, ЕНС ТРУ мен бірлігі бірдей: {len(safe)}")
print("   Сенімді қалпына келетін тауар жолдары:", miss.lot_id.isin(safe.index).sum())

# 2) Лоттың өзінде ЕНС ТРУ сияқты өріс бар ма
lot_any = json.loads(q("SELECT payload FROM raw WHERE entity='lot' LIMIT 1", raw).payload[0])
print("\n2) Лоттағы тауарға қатысты өрістер:",
      {k: v for k, v in lot_any.items() if any(s in k.lower() for s in ("enstru", "ktru", "unit", "tru"))})

# 3) 27 тапсырыс берушінің КАТО-сы subject ішінде бар ма
bins = q("SELECT DISTINCT customer_bin FROM contract_lines").customer_bin.astype(str).tolist()
subj = q("SELECT obj_id, payload FROM raw WHERE entity='subject'", raw)
subj["p"] = subj.payload.map(json.loads)
subj["biin"] = subj.p.map(lambda d: d.get("bin") or d.get("iin") or "")
subj["kato"] = subj.p.map(lambda d: (d.get("kato_list") or [None])[0])
found = subj[subj.biin.isin(bins) | subj.obj_id.astype(str).isin(bins)]
print(f"\n3) Тапсырыс берушілер: {len(bins)}, subject-те табылғаны: {len(found)}")
print(found[["obj_id", "biin", "kato"]].to_string(index=False))

1) Керекті лоттар: 310, ЕНС ТРУ мен бірлігі бірдей: 228
   Сенімді қалпына келетін тауар жолдары: 413

2) Лоттағы тауарға қатысты өрістер: {'enstru_list': [0], 'is_construction_work': 0}

3) Тапсырыс берушілер: 27, subject-те табылғаны: 27
      obj_id         biin      kato
000740001307 000740001307 710000000
020240002363 020240002363 711210000
020440003656 020440003656 711210000
030440003698 030440003698 711210000
050740004819 050740004819 710000000
051040005150 051040005150 117035100
100140011059 100140011059 710000000
120940001946 120940001946 710000000
140340016539 140340016539 710000000
150540000186 150540000186 117035100
171041003124 171041003124 750000000
210240019348 210240019348 711210000
210240033968 210240033968 711210000
210941010761 210941010761 751410000
230740013340 230740013340 710000000
231040023028 231040023028 750000000
780140000023 780140000023 710000000
900640000128 900640000128 710000000
940740000911 940740000911 117035100
940940000384 940940000384 711210000
9604

In [7]:
lp = q("SELECT payload FROM raw WHERE entity='lot'", raw).payload.map(json.loads)
el = lp.map(lambda d: [x for x in (d.get("enstru_list") or []) if x])
print("enstru_list толы лоттар:", (el.str.len() > 0).sum(), "/", len(el))
print(el[el.str.len() > 0].head().tolist())

enstru_list толы лоттар: 946 / 116368
[[1001211], [1000967], [1015744], [1002811], [1001235]]
